# Prompt 3 — Terraform State: Purpose and Management
## Terraform Associate (004) Exam Study Guide

**Exam Objective 2 (continued)**: Understand the purpose of state, what it contains, how to work with it safely, and its security implications.

---

**Topics covered in this notebook:**
1. What is the Terraform State File?
2. Why State is Required
3. What the State File Contains — Annotated JSON Example
4. State and Performance: `-refresh=false`
5. Local State — Default Behavior and Team Limitations
6. Why You Must Never Manually Edit State
7. Sensitive Data in State — Security Implications
8. `terraform state list` — List Tracked Resources
9. `terraform state show` — Inspect a Specific Resource
10. `terraform show` — Human-Readable State or Plan
11. How Terraform Determines Changes: Desired → Current → Actual
12. Exam-Style Questions (3)
13. Real-World Scenarios (5)

---
## 1. What is the Terraform State File?

The **Terraform state file** (`terraform.tfstate`) is a JSON file that Terraform uses to map your HCL configuration to the real-world resources it manages.

```
HCL Configuration      State File              Real Infrastructure
(what you wrote)   ←→  (what Terraform knows)  (what actually exists)

resource            ←→  { "type": "aws_instance",  ←→  EC2 instance
"aws_instance"           "id": "i-0abc123",              i-0abc123
"web" { ... }            "attributes": { ... } }         in us-east-1
```

### The State File's Core Job

Without state, Terraform would have no way to know:
- That `resource "aws_instance" "web"` corresponds to EC2 instance `i-0abc123`
- Whether an existing resource needs to be updated or is already correct
- What resources to destroy when you run `terraform destroy`
- In what order to create resources that depend on each other

### State File Location (Default)

```
project/
├── main.tf
├── variables.tf
├── outputs.tf
└── terraform.tfstate          ← created after first terraform apply
    terraform.tfstate.backup   ← previous state (created before each apply)
```

> **Exam tip**: By default, state is stored **locally** in `terraform.tfstate` in the working directory. For teams, remote backends (S3, Azure Blob, Terraform Cloud) are used instead. Remote backends are covered in a separate objective.

---
## 2. Why State is Required

Terraform requires state for four essential reasons:

### 2.1 Resource Mapping (Identity)

Terraform must know which cloud resource corresponds to which configuration block. Cloud APIs return opaque IDs (like `i-0abc123456789abcd` for EC2). State records this mapping so Terraform can look up, update, and delete the correct resource.

```
HCL: resource "aws_instance" "web"  ──maps to──  i-0abc123456789abcd
```

### 2.2 Computing Diffs (Change Detection)

During `terraform plan`, Terraform compares:
1. **Desired state**: your `.tf` files (what you want)
2. **Last-known state**: `terraform.tfstate` (what Terraform last recorded)
3. **Actual state**: live API calls (what really exists right now)

Without state, Terraform would not know whether a resource already exists or needs to be created.

### 2.3 Dependency Ordering

State records **dependencies** between resources. When creating a subnet that references a VPC, Terraform knows from state that the VPC was created first and stores its ID — so the subnet creation API call can reference the correct VPC ID.

### 2.4 Metadata Storage

State stores provider metadata, resource timeouts, and other information that is not available from the configuration alone. This includes:
- Which provider version created each resource
- Terraform-managed resource lifecycle metadata
- Output values from child modules

### Summary Table

| Reason | Without State | With State |
|--------|---------------|------------|
| Resource mapping | Terraform cannot find existing resources | Maps config names to real resource IDs |
| Change detection | Would recreate everything on every apply | Computes minimal diff — only changes what's needed |
| Dependency ordering | No cross-resource attribute references | Records outputs used by dependent resources |
| Metadata | No provider/version tracking | Records provider version and lifecycle metadata |

---
## 3. What the State File Contains — Annotated JSON Example

Here is a simplified `terraform.tfstate` snippet with annotations explaining each field:

```json
{
  "version": 4,
  "terraform_version": "1.7.3",
  "serial": 12,
  "lineage": "f3a1b2c3-d4e5-6789-abcd-ef0123456789",
  "outputs": {
    "vpc_id": {
      "value": "vpc-0a1b2c3d4e5f6",
      "type": "string",
      "sensitive": false
    }
  },
  "resources": [
    {
      "mode": "managed",
      "type": "aws_vpc",
      "name": "main",
      "provider": "provider[\"registry.terraform.io/hashicorp/aws\"]",
      "instances": [
        {
          "schema_version": 1,
          "attributes": {
            "id": "vpc-0a1b2c3d4e5f6",
            "cidr_block": "10.0.0.0/16",
            "enable_dns_hostnames": true,
            "enable_dns_support": true,
            "tags": {
              "Name": "main-vpc",
              "Environment": "production"
            },
            "arn": "arn:aws:ec2:us-east-1:123456789012:vpc/vpc-0a1b2c3d4e5f6"
          },
          "sensitive_attributes": [],
          "dependencies": []
        }
      ]
    },
    {
      "mode": "managed",
      "type": "aws_subnet",
      "name": "public",
      "provider": "provider[\"registry.terraform.io/hashicorp/aws\"]",
      "instances": [
        {
          "schema_version": 1,
          "attributes": {
            "id": "subnet-0b1c2d3e4f5a",
            "vpc_id": "vpc-0a1b2c3d4e5f6",
            "cidr_block": "10.0.1.0/24",
            "availability_zone": "us-east-1a"
          },
          "sensitive_attributes": [],
          "dependencies": [
            "aws_vpc.main"
          ]
        }
      ]
    }
  ]
}
```

### Field Annotations

| Field | What It Means |
|-------|---------------|
| `version` | State file format version (currently 4) — not the Terraform version |
| `terraform_version` | Version of Terraform that last wrote this state file |
| `serial` | Monotonically incrementing integer — increases with every state write. Used for conflict detection |
| `lineage` | UUID assigned when state is first created — identifies this state lineage uniquely |
| `outputs` | Values from `output {}` blocks — shared across modules and teams |
| `resources[].mode` | `"managed"` (resources you manage) or `"data"` (data sources, read-only) |
| `resources[].type` | The resource type — matches the first label in the `resource {}` block |
| `resources[].name` | The resource name — matches the second label in the `resource {}` block |
| `resources[].provider` | Full provider address — which provider manages this resource |
| `instances[].attributes` | All known attributes of the resource from the last API read |
| `instances[].sensitive_attributes` | Attributes marked sensitive — values are stored but flagged for redaction in output |
| `instances[].dependencies` | Resources this instance depends on — drives creation/destruction ordering |
| `serial` | Increments on each state change — backend uses this to detect concurrent modifications |

---
## 4. State and Performance: `-refresh=false`

### The Refresh Step

By default, `terraform plan` and `terraform apply` perform a **refresh** step before computing the diff:

```
terraform plan steps:
1. Load configuration (.tf files)
2. Load state (terraform.tfstate)
3. REFRESH: Call provider APIs to get current attribute values for each resource
4. Compute diff: desired state vs refreshed actual state
5. Output execution plan
```

The refresh makes one API call per managed resource. With 500+ resources, this can take several minutes.

### `-refresh=false` Flag

```bash
# Skip the refresh step — trust the state file as-is
terraform plan -refresh=false
terraform apply -refresh=false
```

**With `-refresh=false`**:
- Terraform skips all API calls to read current resource attributes
- Uses the state file values directly as the "current state"
- Significantly faster for large configurations
- **Risk**: If someone made out-of-band changes (manual console changes), they will not be detected

### `terraform refresh` (Standalone Command)

```bash
# Update the state file with actual current values — no changes applied
terraform refresh
```

This is equivalent to `terraform apply -refresh-only` in Terraform 1.x+ (preferred modern syntax).

```bash
# Modern preferred approach — shows what would be refreshed before doing it
terraform apply -refresh-only
```

### When to Use Each

| Command | When to Use |
|---------|-------------|
| Default (with refresh) | Normal operations — ensures drift is detected |
| `-refresh=false` | Large configurations where speed matters and you trust the state |
| `terraform apply -refresh-only` | Specifically want to sync state with reality without applying config changes |

---
## 5. Local State — Default Behavior and Team Limitations

### Default: Local State

When no backend is configured, Terraform stores state locally:

```hcl
# No backend block = local state
terraform {
  required_providers {
    aws = { source = "hashicorp/aws", version = "~> 5.0" }
  }
  # No backend block — terraform.tfstate written to working directory
}
```

```
After terraform apply:
project/
├── main.tf
├── terraform.tfstate          ← state file (do NOT commit to Git)
└── terraform.tfstate.backup   ← previous state backup
```

### Local State Limitations for Teams

| Limitation | Problem |
|-----------|--------|
| **Stored on one machine** | Only the engineer who ran `apply` has the state. Others cannot plan/apply without it |
| **No locking** | Two engineers can run `apply` simultaneously — state corruption possible |
| **No versioning** | No history of state changes — cannot roll back to a previous state |
| **Security risk** | State may contain secrets (DB passwords, TLS private keys) — should not be on a laptop |
| **No sharing** | Teams cannot collaborate — each person would need a copy of the state file |
| **Git anti-pattern** | Committing `terraform.tfstate` to Git is strongly discouraged — secrets leak, merge conflicts corrupt state |

### The Solution: Remote Backends

Remote backends (S3 + DynamoDB, Azure Blob Storage, Terraform Cloud, etc.) solve all local state limitations:
- **Shared access**: all team members and CI pipelines read/write the same state
- **Locking**: prevents concurrent applies
- **Versioning**: state history and rollback
- **Encryption**: secrets in state are encrypted at rest

> **Exam tip**: Local state is fine for personal learning but never appropriate for team use. The exam tests that you know **why** remote backends are needed and what problems they solve.

---
## 6. Why You Must Never Manually Edit the State File

### The State File is Terraform's Source of Truth

The `terraform.tfstate` JSON file looks simple enough to edit in a text editor. **Do not do this.**

### Risks of Manual Edits

| Risk | Consequence |
|------|-------------|
| **JSON syntax errors** | Terraform cannot parse the state — all operations fail until fixed |
| **Incorrect resource IDs** | Terraform tries to manage the wrong cloud resource — may destroy the wrong thing |
| **Broken dependencies** | Removing a dependency entry causes out-of-order operations — failures or partial applies |
| **Serial mismatch** | If using remote backends, an incorrect `serial` causes conflict errors |
| **Lineage corruption** | Altering the `lineage` UUID can break state inheritance in workspaces |
| **Lost sensitive attribute markers** | Removing `sensitive_attributes` entries can expose secrets in plan output |

### Use Terraform's State Commands Instead

For any state manipulation, use the proper `terraform state` subcommands:

| Goal | Command | Safe? |
|------|---------|-------|
| Remove a resource from state (without destroying it) | `terraform state rm <address>` | Yes |
| Move a resource to a new address | `terraform state mv <src> <dst>` | Yes |
| Import an existing resource into state | `terraform import <address> <id>` | Yes |
| Replace a resource in state | `terraform state replace-provider` | Yes |
| Edit raw JSON | Manual editing | **NO — never** |

### Emergency State Surgery

If state manipulation is absolutely necessary (e.g., migrating resources between modules):

```bash
# Always create a backup first
cp terraform.tfstate terraform.tfstate.manual-backup-$(date +%Y%m%d)

# Use state commands — safer than manual editing
terraform state mv aws_instance.old_name aws_instance.new_name

# Verify state is intact
terraform plan  # Should show no changes if the rename was all that needed doing
```

> **Exam tip**: The exam may present scenarios asking how to rename a resource without destroying it (`terraform state mv`), or how to remove a resource from management without destroying it (`terraform state rm`). These are safe, supported operations — manual JSON editing is not.

---
## 7. Sensitive Data in State — Security Implications

### What Kind of Secrets End Up in State?

Many Terraform resources store sensitive values as part of their attributes — and **all resource attributes are written to state**:

| Resource | Sensitive Attribute in State |
|----------|-----------------------------|
| `aws_db_instance` | `password` (database password) |
| `aws_iam_access_key` | `secret` (IAM secret access key) |
| `tls_private_key` | `private_key_pem` (TLS private key) |
| `random_password` | `result` (generated password) |
| `azurerm_key_vault_secret` | `value` (the secret value) |
| `kubernetes_secret` | `data` (base64-encoded secret data) |

### The Core Problem

```json
// Inside terraform.tfstate — plaintext secret!
{
  "type": "aws_db_instance",
  "instances": [{
    "attributes": {
      "identifier": "production-db",
      "username": "admin",
      "password": "MySecretPassword123!"
    }
  }]
}
```

Even if you mark a variable as `sensitive = true` in HCL, the value **is still written to state** — it is just not shown in plan/apply output.

### Security Implications

| Risk | Mitigation |
|------|------------|
| **Committed to Git** | Add `*.tfstate` and `*.tfstate.backup` to `.gitignore` |
| **Readable by anyone with state access** | Use IAM/RBAC to restrict who can read the remote backend bucket |
| **Unencrypted at rest** | Enable encryption on the remote backend (S3 server-side encryption, Azure Blob encryption) |
| **Readable in CI logs** | Never print state file contents in CI pipelines |
| **Shared state has broad access** | Consider separate state files per team/environment with different access controls |

### Recommended State Security Pattern

```hcl
# Secure remote backend with encryption and access controls
terraform {
  backend "s3" {
    bucket         = "company-terraform-state"     # Private bucket
    key            = "production/terraform.tfstate"
    region         = "us-east-1"
    encrypt        = true                           # Server-side encryption
    dynamodb_table = "terraform-state-lock"         # State locking

    # Bucket has:
    # - Block all public access: ON
    # - Versioning: enabled (state history)
    # - Access: only Terraform IAM role
    # - KMS encryption for the bucket
  }
}
```

> **Exam tip**: Terraform does not encrypt state automatically — encryption is a feature of the backend storage (S3 KMS, Azure Key Vault-backed Blob, etc.). The `sensitive = true` attribute in HCL only controls output display — it does **not** encrypt or prevent the value from being stored in state.

---
## 8. `terraform state list` — List Tracked Resources

Lists all resources and data sources currently tracked in the state file.

### Syntax

```bash
terraform state list              # List all resources in state
terraform state list aws_instance.*   # Filter by resource type
terraform state list module.vpc.*      # List all resources in a module
```

### Example Output

```bash
$ terraform state list

data.aws_ami.ubuntu
data.aws_availability_zones.available
aws_vpc.main
aws_subnet.public[0]
aws_subnet.public[1]
aws_subnet.private[0]
aws_subnet.private[1]
aws_internet_gateway.main
aws_security_group.web
aws_instance.web[0]
aws_instance.web[1]
aws_instance.web[2]
aws_db_instance.main
module.eks.aws_eks_cluster.main
module.eks.aws_eks_node_group.main
```

### Address Format

Each line is a **resource address** — the fully qualified identifier used in other state commands:

| Address Format | Example | What It Refers To |
|---------------|---------|-------------------|
| `<type>.<name>` | `aws_vpc.main` | Simple resource |
| `<type>.<name>[<index>]` | `aws_subnet.public[0]` | Resource with `count` |
| `<type>.<name>["<key>"]` | `aws_subnet.public["us-east-1a"]` | Resource with `for_each` |
| `data.<type>.<name>` | `data.aws_ami.ubuntu` | Data source |
| `module.<name>.<type>.<name>` | `module.eks.aws_eks_cluster.main` | Resource inside a module |

### Use Cases

- **Audit**: See exactly what Terraform is managing
- **Targeted operations**: Get addresses for `terraform state show`, `terraform taint`, `-target` flags
- **Verification**: After `terraform import`, confirm the resource appears in state
- **Debugging**: Identify orphaned resources or unexpected state entries

---
## 9. `terraform state show` — Inspect a Specific Resource

Shows all attributes of a specific resource as recorded in state — formatted like HCL for readability.

### Syntax

```bash
terraform state show <resource_address>
```

### Example

```bash
$ terraform state show aws_instance.web[0]

# aws_instance.web[0]:
resource "aws_instance" "web" {
    ami                          = "ami-0c55b159cbfafe1f0"
    arn                          = "arn:aws:ec2:us-east-1:123456789012:instance/i-0abc123"
    associate_public_ip_address  = true
    availability_zone            = "us-east-1a"
    cpu_core_count               = 1
    cpu_threads_per_core         = 2
    id                           = "i-0abc123456789abcd"
    instance_state               = "running"
    instance_type                = "t3.micro"
    private_ip                   = "10.0.1.45"
    public_ip                    = "54.210.155.204"
    subnet_id                    = "subnet-0b1c2d3e4f5a"
    tags                         = {
        "Environment" = "production"
        "Name"        = "web-0"
    }
    vpc_security_group_ids       = [
        "sg-0a1b2c3d4e5f",
    ]

    root_block_device {
        delete_on_termination = true
        device_name           = "/dev/sda1"
        iops                  = 100
        volume_id             = "vol-0abc123"
        volume_size           = 20
        volume_type           = "gp2"
    }
}
```

### Key Differences from `terraform state list`

| Command | Output |
|---------|--------|
| `terraform state list` | List of resource addresses (names only) |
| `terraform state show <addr>` | All attributes of one specific resource |
| `terraform show` | All resources in state (or a saved plan) |

### Common Use Cases

- **Debugging**: Check what IP address, ID, or attribute Terraform recorded for a resource
- **Verification after import**: Confirm that `terraform import` captured the right attributes
- **Reference lookup**: Find a resource ID to use in another context (e.g., security group ID for a firewall rule)
- **Sensitive values**: Shows values that are redacted in plan output — use with caution

---
## 10. `terraform show` — Human-Readable State or Plan

`terraform show` renders the current state (or a saved plan file) in a human-readable format.

### Syntax

```bash
terraform show                    # Show current state (all resources)
terraform show -json              # Show current state as JSON
terraform show planfile.tfplan    # Show a saved plan file
terraform show -json planfile.tfplan  # Show a saved plan as JSON (for CI parsing)
```

### Showing Current State

```bash
$ terraform show

# aws_vpc.main:
resource "aws_vpc" "main" {
    arn                              = "arn:aws:ec2:us-east-1:..."
    cidr_block                       = "10.0.0.0/16"
    enable_dns_hostnames             = true
    enable_dns_support               = true
    id                               = "vpc-0a1b2c3d4e5f6"
    ...
}

# aws_subnet.public[0]:
resource "aws_subnet" "public" {
    ...
}
# ... (all resources)

Outputs:

vpc_id = "vpc-0a1b2c3d4e5f6"
```

### Showing a Saved Plan

```bash
# Step 1: Save a plan to file
terraform plan -out=plan.tfplan

# Step 2: Inspect the saved plan
terraform show plan.tfplan

# Step 3 (optional): Get machine-readable JSON of the plan for tooling
terraform show -json plan.tfplan | jq '.resource_changes[] | select(.change.actions[0] == "create")'
```

### `terraform show` vs `terraform state show`

| Command | Purpose | Output |
|---------|---------|--------|
| `terraform show` | Show entire current state OR a plan file | All resources or plan details |
| `terraform state show <addr>` | Show ONE specific resource from state | Attributes of one resource |
| `terraform state list` | List resource addresses in state | Names only (no attribute values) |

---
## 11. How Terraform Determines Changes: Desired → Current → Actual

Understanding the three-way comparison is fundamental to the exam.

### The Three-Way Comparison

```
┌─────────────────────────────────────────────────────────────────┐
│  1. DESIRED STATE                                               │
│     Source: .tf configuration files                             │
│     "I want an EC2 instance t3.micro with tag Name=web"         │
└─────────────────────────────────┬───────────────────────────────┘
                                  │
                                  │  compare
                                  ▼
┌─────────────────────────────────────────────────────────────────┐
│  2. CURRENT STATE (Last Known)                                  │
│     Source: terraform.tfstate                                   │
│     "Last time I checked: i-0abc123 existed, was t3.micro"      │
└─────────────────────────────────┬───────────────────────────────┘
                                  │
                                  │  refresh (API call to verify)
                                  ▼
┌─────────────────────────────────────────────────────────────────┐
│  3. ACTUAL STATE (Real Infrastructure)                          │
│     Source: Live provider API calls                             │
│     "Right now: i-0abc123 exists BUT type is t3.small (drift!)" │
└─────────────────────────────────────────────────────────────────┘
                                  │
                                  │  plan computes diff
                                  ▼
┌─────────────────────────────────────────────────────────────────┐
│  EXECUTION PLAN                                                 │
│  ~ aws_instance.web  # must be updated                         │
│      ~ instance_type = "t3.small" -> "t3.micro"                 │
│  Plan: 0 to add, 1 to change, 0 to destroy.                    │
└─────────────────────────────────────────────────────────────────┘
```

### Decision Matrix: Create, Update, or Destroy?

| Desired (Config) | Current (State) | Actual (API) | Terraform Action |
|-----------------|----------------|--------------|------------------|
| Resource exists | No entry | Does not exist | **Create** the resource |
| Resource exists | Has entry | Exists, matches config | **No change** |
| Resource exists | Has entry | Exists, differs from config | **Update** the resource |
| Resource exists | Has entry | Does not exist (deleted externally) | **Recreate** the resource |
| Resource removed from config | Has entry | Exists | **Destroy** the resource |
| Resource exists | No entry | Exists (not managed by Terraform) | **No action** (use `import` to bring under management) |

### Immutable vs Mutable Updates

Some resource attributes can be **updated in-place** (mutable). Others require the resource to be **destroyed and recreated** (immutable):

```bash
# Mutable attribute — updates in place
~ aws_instance.web
    ~ tags = { "Name" = "old" -> "new" }  # Just an API tag update

# Immutable attribute — forces replacement
-/+ aws_instance.web  # must be replaced
    -/+ ami = "ami-old" -> "ami-new" (forces replacement)
    # Old instance will be destroyed, new one created
```

When a replacement is required, Terraform shows `-/+` in the plan. This is important to notice before applying — it means downtime for stateful resources like databases.

> **Exam tip**: Terraform reads both the state file AND makes live API calls during `plan` (unless `-refresh=false`). State alone is not enough — the actual infrastructure may have drifted from what state recorded.

---
## 12. Exam-Style Practice Questions

---

**Q1: An engineer manually terminates an EC2 instance managed by Terraform using the AWS console. The next time a team member runs `terraform plan`, what will Terraform report?**

A) No changes — Terraform does not detect externally-made changes  
B) An error — Terraform cannot continue because the state file is corrupt  
C) That the EC2 instance will be created — because the resource exists in configuration but no longer exists in actual infrastructure  
D) That the EC2 instance will be imported — Terraform automatically imports recreated resources  

<details>
<summary>Answer</summary>

**Answer: C**  
During `terraform plan`, Terraform refreshes the state by calling the AWS API. It discovers that `i-0abc123` no longer exists. Since the resource is still declared in the `.tf` configuration, Terraform plans to recreate it (`+` action). Option A is wrong — Terraform detects drift via API refresh. Option B is wrong — the state file is valid; the resource is just gone. Option D is wrong — Terraform does not auto-import.

</details>

---

**Q2: Which of the following statements about the Terraform state file is true?**

A) Sensitive values marked with `sensitive = true` in HCL are not written to the state file  
B) The state file should be committed to version control to share it across team members  
C) The state file may contain plaintext secrets such as database passwords and private keys  
D) The state file is automatically encrypted by Terraform when it is written to disk  

<details>
<summary>Answer</summary>

**Answer: C**  
All resource attributes — including sensitive values — are written to the state file in plaintext. The `sensitive = true` annotation only prevents the value from appearing in plan/apply terminal output; it does not encrypt or omit the value from state. Option A is wrong for this reason. Option B is wrong — state should never be committed to Git (add to .gitignore). Option D is wrong — Terraform does not encrypt state; that is a backend feature.

</details>

---

**Q3: Which command would you use to see all attributes (including the resource ID, private IP, and all tags) of a specific EC2 instance managed by Terraform, as recorded in the state file?**

A) `terraform state list aws_instance.web`  
B) `terraform state show aws_instance.web`  
C) `terraform plan -target=aws_instance.web`  
D) `terraform output aws_instance.web`  

<details>
<summary>Answer</summary>

**Answer: B**  
`terraform state show <address>` displays all recorded attributes for a specific resource in a human-readable HCL-like format. Option A (`state list`) only shows the resource address — no attribute values. Option C (`plan -target`) would show a plan diff, not current state values. Option D (`output`) is for declared output blocks, not individual resource attributes.

</details>

---
## 13. Real-World Scenarios

<details>
<summary>Scenario 1 — State File Accidentally Committed to Git (Secrets Leaked)</summary>

**Situation:**
A junior developer on a new team creates their first Terraform configuration, runs `terraform apply`, and then does `git add . && git commit -m 'initial infrastructure'`. They accidentally commit `terraform.tfstate` to the company's GitHub repository. The state file contains the RDS database password and an IAM access key secret in plaintext.

**Immediate remediation:**

```bash
# Step 1: Rotate compromised credentials immediately
# Rotate RDS password:
aws rds modify-db-instance --db-instance-identifier prod-db \
  --master-user-password "NewSecurePassword456!"

# Rotate IAM access key:
aws iam delete-access-key --access-key-id AKIA...
aws iam create-access-key --user-name terraform-ci

# Step 2: Remove state file from Git history
git filter-repo --path terraform.tfstate --invert-paths
# Or using BFG Repo Cleaner:
# bfg --delete-files terraform.tfstate
git push --force-with-lease origin main

# Step 3: Add to .gitignore to prevent recurrence
echo '*.tfstate' >> .gitignore
echo '*.tfstate.backup' >> .gitignore
git add .gitignore
git commit -m 'chore: add terraform state files to gitignore'

# Step 4: Migrate to remote backend
# Add S3 backend to prevent local state in the future
```

**Correct .gitignore:**
```gitignore
.terraform/
*.tfstate
*.tfstate.backup
*.tfvars       # if containing secrets
crash.log
override.tf
```

**Expected outcome:**
- Compromised credentials rotated before exploitation
- State file history purged from Git
- `.gitignore` prevents future accidents
- Team migrated to remote backend with encryption

**Exam sub-objective demonstrated:** Sensitive data in state, local state limitations, `.gitignore` for state files.

</details>

<details>
<summary>Scenario 2 — Detecting and Reconciling Infrastructure Drift</summary>

**Situation:**
An operations engineer receives an urgent incident alert. They log into the AWS console and manually change a production security group to allow port 5432 from `0.0.0.0/0` (all internet traffic to Postgres). The incident is resolved, but they forget to update the Terraform configuration or remove the temporary rule. Three days later, a security scan flags the open Postgres port.

**Using `terraform plan` to detect drift:**

```bash
$ terraform plan

Terraform will perform the following actions:

  # aws_security_group.database will be updated in-place
  ~ resource "aws_security_group" "database" {
      # (1 unchanged attribute hidden)

      - ingress {
          - cidr_blocks = [
              - "0.0.0.0/0",
            ]
          - from_port   = 5432
          - protocol    = "tcp"
          - to_port     = 5432
        }
    }

Plan: 0 to add, 1 to change, 0 to destroy.

# The plan shows the unauthorized rule that needs to be removed
```

**Options for resolution:**

```bash
# Option A: Remove the drift by applying (restores to desired state)
terraform apply

# Option B: Legitimise the change by updating the config
# Add to main.tf:
#   ingress {
#     from_port   = 5432
#     to_port     = 5432
#     protocol    = "tcp"
#     cidr_blocks = [var.db_allowed_cidr]  # Restrict to specific CIDR
#   }
# Then: terraform plan && terraform apply

# Option C: Sync state only (update state to match reality without changing config)
terraform apply -refresh-only
# This updates state to include the new rule without changing infrastructure
# But the rule remains open in production!
```

**Expected outcome:**
- Drift detected automatically via `terraform plan`
- Security team can see the exact change that was made
- Resolution is deliberate and documented via a PR

**Exam sub-objective demonstrated:** Desired vs actual state reconciliation, `-refresh-only`, drift detection.

</details>

<details>
<summary>Scenario 3 — Importing Existing Infrastructure into State</summary>

**Situation:**
A company has been running production infrastructure for 2 years — all created manually through the AWS console. They want to adopt Terraform but cannot afford downtime to recreate everything. They need to bring existing resources under Terraform management without destroying and recreating them.

**Using `terraform import` to adopt existing resources:**

```hcl
# Step 1: Write the configuration for the existing VPC
# main.tf
resource "aws_vpc" "main" {
  cidr_block           = "10.0.0.0/16"
  enable_dns_hostnames = true
  enable_dns_support   = true

  tags = {
    Name = "production-vpc"
  }
}
```

```bash
# Step 2: Import the existing VPC into state
# The VPC ID is found in the AWS console
terraform import aws_vpc.main vpc-0a1b2c3d4e5f6a7b8

# Output:
# aws_vpc.main: Importing from ID "vpc-0a1b2c3d4e5f6a7b8"...
# aws_vpc.main: Import prepared!
#   Prepared aws_vpc for import
# aws_vpc.main: Refreshing state... [id=vpc-0a1b2c3d4e5f6a7b8]
# Import successful!
# The resources that were imported are shown above. These resources are now in
# your Terraform state and will henceforth be managed by Terraform.

# Step 3: Verify the import
terraform state show aws_vpc.main
# Check that all attributes match the configuration

# Step 4: Run plan — should show no changes if config matches reality
terraform plan
# Plan: 0 to add, 0 to change, 0 to destroy.  ← ideal outcome
# If changes are shown, update the config to match the real resource attributes
```

**Expected outcome:**
- Existing VPC is now tracked in state without recreation
- Future changes to the VPC go through Terraform plan/apply
- `terraform plan` shows 0 changes when configuration matches reality
- Team gradually imports all existing resources over a migration period

**Exam sub-objective demonstrated:** `terraform import`, state mapping, `terraform state show` verification.

</details>

<details>
<summary>Scenario 4 — Resource Rename Requires `terraform state mv`</summary>

**Situation:**
A developer refactors their Terraform configuration and renames `resource "aws_s3_bucket" "data"` to `resource "aws_s3_bucket" "data_lake"` to better describe its purpose. Running `terraform plan` immediately shows Terraform wants to destroy the old bucket and create a new one — which would cause data loss.

**The problem:**

```bash
$ terraform plan

  # aws_s3_bucket.data will be destroyed
  - resource "aws_s3_bucket" "data" {
      - bucket = "company-data-lake-prod"
      ...
    }

  # aws_s3_bucket.data_lake will be created
  + resource "aws_s3_bucket" "data_lake" {
      + bucket = "company-data-lake-prod"
      ...
    }

Plan: 1 to add, 0 to change, 1 to destroy.  ← DATA LOSS RISK!
```

**Solution: `terraform state mv`**

```bash
# Rename the resource in state to match the new config name
# WITHOUT touching actual infrastructure
terraform state mv aws_s3_bucket.data aws_s3_bucket.data_lake

# Output:
# Move "aws_s3_bucket.data" to "aws_s3_bucket.data_lake"
# Successfully moved 1 object(s).

# Now plan shows no changes
terraform plan
# Plan: 0 to add, 0 to change, 0 to destroy.  ← safe!
```

**Expected outcome:**
- S3 bucket is not destroyed — only its state address is renamed
- `terraform plan` shows no infrastructure changes
- The refactored name is now canonical in both config and state

**Exam sub-objective demonstrated:** `terraform state mv`, why not to manually edit state, resource address format.

</details>

<details>
<summary>Scenario 5 — Performance Optimization with `-refresh=false` in Large Workspaces</summary>

**Situation:**
An engineering team manages a large production workspace with 800+ Terraform-managed resources across AWS, Datadog, and GitHub. Every `terraform plan` takes 12 minutes because Terraform makes API calls for all 800 resources during the refresh step. The CI pipeline runs plan on every PR, and developers are frustrated by the slow feedback loop.

**Analysis of the bottleneck:**

```bash
$ time terraform plan
# ...
# real    12m34s   ← 12 minutes just for plan!

# With timing breakdown:
# Refresh phase: ~11 minutes (800 API calls, rate-limited by providers)
# Plan computation: ~30 seconds
```

**Solution using `-refresh=false` in CI:**

```yaml
# .github/workflows/terraform.yml
jobs:
  plan:
    steps:
      - name: Terraform Plan (fast - no refresh)
        run: terraform plan -refresh=false -out=plan.tfplan
        # Drops from 12 minutes to 45 seconds
        # Acceptable because:
        # 1. PRs change configuration, not out-of-band infrastructure
        # 2. Drift detection runs on a separate scheduled job

  drift-detection:
    # Run on a schedule, not on every PR
    schedule: '0 6 * * *'  # 6am daily
    steps:
      - name: Detect Drift (with full refresh)
        run: terraform plan  # Full refresh — catches any manual console changes
```

**State list for targeted operations:**

```bash
# When a specific component changes, use -target to limit scope
terraform state list | grep 'module.api'
# module.api.aws_lambda_function.handler
# module.api.aws_api_gateway_rest_api.main
# module.api.aws_iam_role.lambda_exec

# Target plan to only the changed module — much faster
terraform plan -target=module.api
```

**Expected outcome:**
- PR pipeline: 45 seconds instead of 12 minutes
- Drift still detected daily via scheduled full-refresh plan
- Developer feedback loop dramatically improved

**Exam sub-objective demonstrated:** `-refresh=false` performance optimization, `terraform state list` for targeted operations, understanding the refresh step.

</details>

---
## Quick-Reference Cheat Sheet

```
Terraform State Cheat Sheet
═══════════════════════════════════════════════════════════════════

WHAT IS STATE?
  JSON file mapping HCL config → real resource IDs + attributes
  Default: terraform.tfstate in working directory
  Backup:  terraform.tfstate.backup (previous state)

WHY STATE IS NEEDED
  1. Resource mapping (config name → cloud resource ID)
  2. Change detection (desired vs actual diff)
  3. Dependency ordering (records cross-resource references)
  4. Metadata storage (provider versions, lifecycle)

THREE-WAY COMPARISON (during plan)
  Desired State  (.tf files)      ← what you want
  Current State  (tfstate)        ← what Terraform last saw
  Actual State   (live API calls) ← what really exists
  → Diff = execution plan

STATE COMMANDS
  terraform state list              → list all resource addresses
  terraform state show <addr>       → all attributes of one resource
  terraform show                    → all resources in state
  terraform show <planfile>         → inspect saved plan
  terraform state mv <src> <dst>    → rename resource in state
  terraform state rm <addr>         → remove from state (no destroy)
  terraform import <addr> <id>      → import existing resource

PERFORMANCE
  terraform plan -refresh=false     → skip API calls (trust state)
  terraform apply -refresh-only     → sync state, no config changes

SECURITY
  ❌ Secrets ARE in state (plaintext) even if marked sensitive
  ❌ Never commit *.tfstate to Git
  ❌ Never manually edit the state file JSON
  ✅ Use remote backend with encryption + access controls
  ✅ sensitive = true hides in output only (NOT from state)

LOCAL STATE LIMITATIONS (why teams need remote backends)
  - Only on one machine
  - No locking (concurrent apply = state corruption)
  - No versioning or rollback
  - Secrets on local disk
═══════════════════════════════════════════════════════════════════
```